###
<h1><strong>Task 1: Predictive Modeling (Classification)</strong></h1>
<p><strong>Description:</strong> Build and evaluate a classification model to predict categorical outcomes (e.g., predict if a customer will churn).</p>

<h3><strong> Objectives:</strong></h3> 
<ul>
    <li>Preprocess the data (handle categorical variables,feature scaling).</li>
    <li>Train and test multiple classification models (e.g., Decision Trees, Logistic Regression, Random Forest).</li>
    <li>Evaluate models using accuracy, precision, recall, and F1-score.</li>
    <li>Perform hyperparameter tuning using grid search.</li>
</ul>
<p>Tools: Python, scikit-learn, pandas, matplotlib.</p>


In [2]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

CLEAN_DIR = PROJECT_ROOT / "data/cleaned_data"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projet : {PROJECT_ROOT}")
print(f"Cleaned files : {len(list(CLEAN_DIR.rglob('*.csv')))}")

Projet : /home/broman/Downloads/CodVeva_internship/codveda_datanalysis
Cleaned files : 5


<h4>The dataset: Churn</h4>
<p>
    <h4><strong><li>Data preprocessing</li></strong></h4>
</p>

In [3]:
churn_data = pd.read_csv(CLEAN_DIR / "c_churn_combined.csv")
churn_data.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,LA,117,408,No,No,0,184.5,97,31.37,351.6,80,29.89,215.8,90,9.71,8.7,4,2.35,1,False
1,IN,65,415,No,No,0,129.1,137,21.95,228.5,83,19.42,208.8,111,9.40,12.7,6,3.43,4,True
2,NY,161,415,No,No,0,332.9,67,56.59,317.8,97,27.01,160.6,128,7.23,5.4,9,1.46,4,True
3,SC,111,415,No,No,0,110.4,103,18.77,137.3,102,11.67,189.6,105,8.53,7.7,6,2.08,2,False
4,HI,49,510,No,No,0,119.3,117,20.28,215.1,109,18.28,178.7,90,8.04,11.1,1,3.00,1,False


In [4]:
#Identifying categorical variables

categorical_vars = [
    "State",
    "Area code",
    "International plan",
    "Voice mail plan",
    "Churn",
]

numeric_vars = [
    column for column in churn_data.columns
    if column not in categorical_vars
]

print("Categorical Variables :")
display(categorical_vars)

print("\nNumeric Variables :")
display(numeric_vars)

#Feature scaling 
scaler = StandardScaler()
churn_data[numeric_vars] = scaler.fit_transform(churn_data[numeric_vars]) 


Categorical Variables :


['State', 'Area code', 'International plan', 'Voice mail plan', 'Churn']


Numeric Variables :


['Account length',
 'Number vmail messages',
 'Total day minutes',
 'Total day calls',
 'Total day charge',
 'Total eve minutes',
 'Total eve calls',
 'Total eve charge',
 'Total night minutes',
 'Total night calls',
 'Total night charge',
 'Total intl minutes',
 'Total intl calls',
 'Total intl charge',
 'Customer service calls']

### 1. Decision Trees Classification

In [5]:
# We split the categorical variables according to the best encoding strategy

cat_nom_cols = ['Area code', 'International plan', 'Voice mail plan'] # One-Hot encoding
cat_high_card = ['State'] # High cardinality -> Target Encoding

# We split the features and the target
X = churn_data[numeric_vars + cat_nom_cols + cat_high_card]
# Encoding the target variable to 0 and 1 as it is in text format ("Yes"/"No")
y = churn_data['Churn'].map({'Yes': 1, 'No': 0}) if churn_data['Churn'].dtype == 'object' else churn_data['Churn']

# Creation of preprocessing pipelines
num_transformer = SimpleImputer(strategy='median')

nom_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

state_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_enc', TargetEncoder(cv=5)) # Encoding according to the churn rate by state
])

# Final embedding of the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric_vars),
        ('nom', nom_transformer, cat_nom_cols),
        ('state', state_transformer, cat_high_card)
    ]
)

# Train split / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Application of the preprocessing
X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Our data are now ready for the DecisionTreeClassifier !
print("Preprocessing successful. Final number of features :", X_train_preprocessed.shape[1])


# We train a decision tree model to predict churn based on the features in the dataset. We will use the categorical variables as dummy variables and the numeric variables as they are.
 

# We  train a logistic regression model to predict churn based on the features in the dataset. We will use the categorical variables as dummy variables and the numeric variables as they are.

Preprocessing successful. Final number of features : 23


In [6]:
# We initialize and fit the Decision Tree
# Note: Limiting max_depth prevents the tree from over-focusing on noise
classif_tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
classif_tree_model.fit(X_train_preprocessed, y_train)

# We extract and rank the chosen features
importances = classif_tree_model.feature_importances_
feature_names = X.columns

# A clean DataFrame of the results
feature_names = preprocessor.get_feature_names_out()
feature_rank = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_rank)

                        Feature  Importance
2        num__Total day minutes    0.313669
14  num__Customer service calls    0.151494
11      num__Total intl minutes    0.117456
12        num__Total intl calls    0.101784
19  nom__International plan_Yes    0.098961
5        num__Total eve minutes    0.095370
21     nom__Voice mail plan_Yes    0.063244
10      num__Total night charge    0.016222
7         num__Total eve charge    0.010973
3          num__Total day calls    0.009186
18   nom__International plan_No    0.008295
4         num__Total day charge    0.007280
13       num__Total intl charge    0.006066
0           num__Account length    0.000000
1    num__Number vmail messages    0.000000
6          num__Total eve calls    0.000000
9        num__Total night calls    0.000000
8      num__Total night minutes    0.000000
15           nom__Area code_408    0.000000
17           nom__Area code_510    0.000000
16           nom__Area code_415    0.000000
20      nom__Voice mail plan_No 

In [7]:
# We generate predictions
y_pred_train_tree = classif_tree_model.predict(X_train_preprocessed)
y_pred_test_tree = classif_tree_model.predict(X_test_preprocessed)

### 2. Logistic Regression Classification

In [8]:

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = numeric_vars
categorical_features = [col for col in categorical_vars if col != 'Churn']  # Exclude the target variable

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train_processed, y_train)
y_pred_train_logreg = log_reg_model.predict(X_train_processed)
y_pred_test_logreg = log_reg_model.predict(X_test_processed)

### 3. Random Forests Classification

In [9]:

from sklearn.ensemble import RandomForestClassifier


rand_model = RandomForestClassifier(n_estimators=100, random_state=42)
rand_model.fit(X_train_processed, y_train)
y_pred_train_rand = rand_model.predict(X_train_processed)
y_pred_test_rand = rand_model.predict(X_test_processed)

### Evaluate models using accuracy, precision, recall, and F1-score.

#### 1. Decision Trees

In [10]:
#  We calculate each metric individually
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score


accuracy = accuracy_score(y_test, y_pred_test_tree)
precision = precision_score(y_test, y_pred_test_tree) # Basically for the positive class (1 / Churn)
recall = recall_score(y_test, y_pred_test_tree) # Basically for the positive class (1 / Churn)

display(pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall'],
    'Value': [accuracy, precision, recall]
}))

# Then print the classification report (contain the F1-score)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred_test_tree, target_names=['Faith (0)', 'Churn (1)']))



,Metric,Value
0,Accuracy,0.932534
1,Precision,0.814815
2,Recall,0.687500


--- Classification Report ---
              precision    recall  f1-score   support

   Faith (0)       0.95      0.97      0.96       571
   Churn (1)       0.81      0.69      0.75        96

    accuracy                           0.93       667
   macro avg       0.88      0.83      0.85       667
weighted avg       0.93      0.93      0.93       667



#### 2. Logistic Regression 

In [11]:
accuracy = accuracy_score(y_test, y_pred_test_logreg)
precision = precision_score(y_test, y_pred_test_logreg) # Basically for the positive class (1 / Churn)
recall = recall_score(y_test, y_pred_test_logreg) # Basically for the positive class (1 / Churn)

display(pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall'],
    'Value': [accuracy, precision, recall]
}))

# Then print the classification report (contain the F1-score)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred_test_logreg, target_names=['Faith (0)', 'Churn (1)']))


,Metric,Value
0,Accuracy,0.860570
1,Precision,0.538462
2,Recall,0.218750


--- Classification Report ---
              precision    recall  f1-score   support

   Faith (0)       0.88      0.97      0.92       571
   Churn (1)       0.54      0.22      0.31        96

    accuracy                           0.86       667
   macro avg       0.71      0.59      0.62       667
weighted avg       0.83      0.86      0.83       667



#### 3. Random Forests

In [12]:
accuracy = accuracy_score(y_test, y_pred_test_rand)
precision = precision_score(y_test, y_pred_test_rand) # Basically for the positive class (1 / Churn)
recall = recall_score(y_test, y_pred_test_rand) # Basically for the positive class (1 / Churn)

display(pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall'],
    'Value': [accuracy, precision, recall]
}))

# Then print the classification report (contain the F1-score)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred_test_rand, target_names=['Faith (0)', 'Churn (1)']))


,Metric,Value
0,Accuracy,0.941529
1,Precision,0.952381
2,Recall,0.625000


--- Classification Report ---
              precision    recall  f1-score   support

   Faith (0)       0.94      0.99      0.97       571
   Churn (1)       0.95      0.62      0.75        96

    accuracy                           0.94       667
   macro avg       0.95      0.81      0.86       667
weighted avg       0.94      0.94      0.94       667



### Performing hyper parameter tuning

Hyper-parameter are the parameters that are set before starting the training of the model. (i.e. Learning Rate, regularization parameter)
Model Parameters are the ones' that are learned while model training. (i.e. Weights of Neural Network)


In [21]:
#Performing hyperparameter tuning for the decision trees model using GridSearchCV

from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
#Search for the best hyperparameters using GridSearchCV
grid_search = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_preprocessed, y_train)
display(pd.DataFrame({
    'Max Depth': [grid_search.best_params_['max_depth']], 
    'Min Samples Split': [grid_search.best_params_['min_samples_split']],
    'Min Samples Leaf': [grid_search.best_params_['min_samples_leaf']],
    
    'Best Cross-Validation Score': [grid_search.best_score_]
}).rename(index={0: 'Best values'}))

,Max Depth,Min Samples Split,Min Samples Leaf,Best Cross-Validation Score
Best values,7,2,2,0.937738


In [34]:
# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred_test_best_tree = best_model.predict(X_test_preprocessed)
test_accuracy = accuracy_score(y_test, y_pred_test_best_tree)
print(f'Test Set Accuracy with Tuned Decision Tree Model: {test_accuracy:.4f}')

Test Set Accuracy with Tuned Decision Tree Model: 0.9295


In [ ]:
#Performing hyperparameter tuning for the logistic regression model using GridSearchCV

pipeline = Pipeline([
    ('model', LogisticRegression(
        solver='saga', #Supports all penalties (l1, l2, elasticnet, and none) and is efficient for larger datasets.
        max_iter=8000, #The maximum number of iterations taken for the solver to converge to a solution. 
        tol=1e-4,
        random_state=42,
    ))
])

param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],  
    'model__l1_ratio': [0, 0.5, 1]
}

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='accuracy',error_score='raise')
grid_search.fit(X_train_processed, y_train)
display(pd.DataFrame({
    'C': [grid_search.best_params_['model__C']],
    'L1 Ratio': [grid_search.best_params_['model__l1_ratio']],
    'Solver': [grid_search.best_params_['model']],
    'Best Cross-Validation Score': [grid_search.best_score_]
}).rename(index={0: 'Best values'}))


/home/broman/Downloads/CodVeva_internship/codveda_datanalysis/.veda_venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/broman/Downloads/CodVeva_internship/codveda_datanalysis/.veda_venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/broman/Downloads/CodVeva_internship/codveda_datanalysis/.veda_venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/broman/Downloads/CodVeva_internship/codveda_datanalysis/.veda_venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/broman/Downloads/CodVeva_internship/codveda_da

KeyboardInterrupt: 

In [37]:
#Performing hyperparameter tuning for the random forest model using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
#Search for the best hyperparameters using GridSearchCV
grid_search_rand = GridSearchCV(estimator=RandomForestClassifier(),param_grid= param_grid, cv=5, verbose=1,scoring='accuracy')
grid_search_rand.fit(X_train_processed, y_train)
display(pd.DataFrame({
    'N Estimators': [grid_search_rand.best_params_['n_estimators']],
    'Max Depth': [grid_search_rand.best_params_['max_depth']],
    'Min Samples Split': [grid_search_rand.best_params_['min_samples_split']], 
    'Min Samples Leaf': [grid_search_rand.best_params_['min_samples_leaf']],
    'Best Cross-Validation Score': [grid_search_rand.best_score_]
}).rename(index={0: 'Best values'}))


Fitting 5 folds for each of 81 candidates, totalling 405 fits


,N Estimators,Max Depth,Min Samples Split,Min Samples Leaf,Best Cross-Validation Score
Best values,100,None,2,1,0.942612


In [42]:
# Evaluate the best model on the test set
best_model = grid_search_rand.best_estimator_
y_pred_test_best_tree = best_model.predict(X_test_preprocessed)
test_accuracy = accuracy_score(y_test, y_pred_test_best_tree)
print(f'Test Set Accuracy with Tuned Random Forest Model: {test_accuracy:.4f}')

ValueError: X has 23 features, but RandomForestClassifier is expecting 73 features as input.